# FDM modeling of ARK-cross sections

We use the developed code from "src/ARK_geotop.py"

In [ ]:
import os
import sys
from typing import Any
from glob import glob
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import pdf2image
import pickle

from tools.fdm.src.mfgrid import Grid
from tools.fdm.src.fdm3blom import Fdm3

from mf6lab.Projects.ARK_RWS.src.ARK_geotop import (
    sinspace,
    parse_geotop_filename,
    Geotop_xsec,
    CrossSectionDigitizer,
    ImagePicker,
    Dirs,
    plot_result,    
    LITHO_CLASSES,
    GEO_UNITS
    )

print(sys.executable)

# --- Needed to make figure separate from the notebook and interactive
%matplotlib qt

# --- Seet the namespace for the relevant directories
dirs = Dirs()

# --- Get the paths and names of  the geotop pdf files in the order they are in dirs.dino
xsec_paths = {i:name for i, name in enumerate(glob(dirs.dino + '*.pdf'))}
xsec_names = {i:os.path.basename(name) for i, name in enumerate(glob(dirs.dino + '*.pdf'))}

# --- Pickling
def pickleto(var:Any, basename:str, parent:str=dirs.data):
    """Pickle var to os.path.join(dirs.data, basename)"""
    if not basename.endswith('.pkl'):
        basename += ".pkl"

    pkl_file = os.path.join(parent, basename)
    with open(pkl_file, 'wb') as f:
        print(f"Pickled {basename} --> {parent}")        
        pickle.dump(var, f)

# --- Unpickling
def picklefrom(basename:str, parent:str=dirs.data)->Any:
    """Unpickle varname from os.path.join(parent, basename)"""
    if not basename.endswith(".pkl"):
        basename += ".pkl"
          
    pkl_file = os.path.join(parent, basename)
    with open(pkl_file, 'rb') as f:
        print(f"Loaded {basename} <-- {parent}")        
        return pickle.load(f)


def spy(idx_arr):
    """Show where the index labels are in the xsec idx_arr."""
    fig, ax = plt.subplots(figsize=(10, 6))    
    ax.set_title("Location of legend indices in xsec.idx_arr")
    classes = np.unique(idx_arr)
    cmap = plt.get_cmap('tab20', len(classes) - 1)
    mappable = ax.imshow(idx_arr, cmap=cmap, origin='upper')
    fig.colorbar(mappable)
    plt.show()

# --- Color for empty legend (empty voxel with geo_unit 'none')
WHITE_01 = np.array([1., 1., 1.])

loading mfpath.py
/Users/Theo/Development/python/mf6_tools/mf6lab/.venv/bin/python


In [2]:
xsec_names

{0: 'BRO GeoTOP Verticale doorsnede geologische eenheid 130173,479431.pdf',
 1: 'BRO GeoTOP Verticale doorsnede meest waarschijnlijke lithoklasse 126311,474187.pdf',
 2: 'BRO GeoTOP Verticale doorsnede meest waarschijnlijke lithoklasse 130049,479466.pdf',
 3: 'BRO GeoTOP Verticale doorsnede geologische eenheid 129926,479462.pdf',
 4: 'BRO GeoTOP Verticale doorsnede geologische eenheid 127484,477893.pdf',
 5: 'BRO GeoTOP Verticale doorsnede geologische eenheid 130049,479466.pdf',
 6: 'BRO GeoTOP Verticale doorsnede geologische eenheid 126311,474187.pdf',
 7: 'BRO GeoTOP Verticale doorsnede meest waarschijnlijke lithoklasse 129926,479462.pdf',
 8: 'BRO GeoTOP Verticale doorsnede meest waarschijnlijke lithoklasse 127484,477893.pdf',
 9: 'BRO GeoTOP Verticale doorsnede meest waarschijnlijke lithoklasse 130173,479431.pdf'}

In [3]:
geotop_xsecs = picklefrom("geotop_xsecs.pkl")

Loaded geotop_xsecs.pkl <-- /Users/Theo/Development/python/mf6_tools/mf6lab/Projects/ARK_RWS/data/


## Deal with the second cross section only

In [5]:
isec = 1
xsec = geotop_xsecs[xsec_names[isec]]

print(f"Dealing with xsec {isec}:\n{xsec.name}") 

# --- find the ARK it wronly has index 1 in row 6
ix_ARK = np.where(xsec.idx_arr[6] == 1)[0]

spy(xsec.idx_arr)


Dealing with xsec 1:
BRO GeoTOP Verticale doorsnede meest waarschijnlijke lithoklasse 126311,474187.pdf


### Repair the idx_arr for the ARK canal which looks now filled with material 'a'

The problem is that the idx_arr contains index 1 (anthropogenic) inside the ARK,
which was likely caused by some vertical grid line disturbing the color_match,
so the light gray legend color (1) was matched instead of white (0).

To find ARK look for index = 1 in row 6 to find ARK incision in the xsec.
Then replace all the 1 indices in that column with 0 to make it empty.

In [6]:
idx_arr = xsec.idx_arr.copy()

# --- The columns have idx 1 incorrectly
cols = np.where(idx_arr[1] == 1)[0]

# --- Replace by index 0 ('none')
for j in cols:
    rows = idx_arr[:, j] == 1
    idx_arr[rows, j] = 0
    
# --- Check
spy(idx_arr)

When this works replace the xsec.idx_arr with the corrected version

In [7]:
# --- Replace origional idx_arr by corrected one
xsec.idx_arr = idx_arr

# --- Check
spy(xsec.idx_arr)
print('idx_arr repaired')

idx_arr repaired


## Model grid

In [8]:
def Ifrom_v_extent(gr, extent):
    """Return global index given vertical extent = (xmin, xmax, zmin, zmax) """
    xmin, xmax, zmin, zmax = extent
    mask = np.logical_and.reduce([
        gr.XM > xmin, gr.XM < xmax,
        gr.ZM > zmin, gr.ZM < zmax
    ])
    return gr.NOD[mask]


In [9]:
def sinspace(L, n):
    z = np.cumsum(np.sin(np.linspace(0, np.pi / 2, n)))
    z = L * z / z[-1]
    return z


In [10]:
sinspace(100, 10)

array([  0.        ,   2.79400558,   8.2971223 ,  16.34214074,
        26.68461709,  39.01030044,  52.94468112,  68.06437005,
        83.90996312, 100.        ])

In [ ]:
def set_ARK_xsec(xsec):
    xsec.ground_elev = -1.3,
    xsec.stage  = -0.4
    xsec.z_bot  = -6.0
    xsec.d_damw = 0.5,
    xsec.z_damw = -12.,
    xsec.hpp    = xsec.stage - 1,
    xsec.c_drainage = 100.,
            
    # --- where in the cross section is the ARK
    ixARK = np.where(xsec.idx_arr[10] == 0)[0][0]
    
    # --- ARK water extent
    ae = np.array([xsec.x[ixARK], xsec.x[ixARK+1], xsec.z_bot, xsec.stage])
    
    # --- Half width of ARK:
    xsec.b = (ae[1] - ae[0]) / 2

    # --- Centralize xsec.x around xARKmid
    xsec.xARKmid_orig     = 0.5 * (ae[0] + ae[1])
    xsec.world_extent[:2] -= xsec.xARK_orig
    
    # --- extent of the sheet piling left and right along canal      
    xsec.damwL = np.array([-xsec.b - xsec.d_damw, -xsec.b, xsec.z_damw, 0])
    xsec.damwR = np.array([+xsec.b, +xsec.b + xsec.d_damw, xsec.z_damw, 0])


    # --- Grid x-coordinates (parallel to xsec) with 0 at xsec.xARK_orig
    # --- refined around the edges of the canal xsec.  
    slog = np.logspace(0, 2, 10)

    # --- Horizontal grid refined incide and next to the Canal.
    b = xsec.b

    x_ = np.hstack((
        0, xsec.damwR[:2],
        (b - np.logspace(0, np.log10(b), 10))[::-1], # --- inside ARK, right of middle
        xsec.d_damw[:2],                                # --- sheet piling
        b + slog[slog > xsec.d_damw],                   # --- increasing cell wirdth first 100 m 
        np.linspace(0, 2000, 21).clip(200, None)        # --- beyond this to 2000 m 100 m cells
    )).clip(5., None)

    # --- Mirror around heart line of ARK and remove doubles   
    x = np.unique(np.hstack((-x_[::-1], x_)))
    
    # --- The grid use the new x and the old z
    xsec.gr = Grid(x, None, xsec.z[xsec.z <= -1.3])
    
    
    # --- Drainage        
    drn_ext = np.array([xsec.gr.x[0], xsec.gr.x[-1], xsec.hpp - 0.25, xsec.hpp + 0.25])
    drn_mask = np.logical_and(
         xsec.gr.inblock(xx=drn_ext[:2], yy=None, zz=drn_ext[-2:]),
        ~xsec.gr.inblock(xx=ae[:2], yy=None, zz=ae[-2:]))
    
    xsec.Idrn = xsec.gr.NOD[drn_mask]

    DRN = np.zeros(len(xsec.Idrn), dtype=Fdm3.dtype['drn'])
    DRN['Ig',:] = xsec.Idrn
    DRN['C'] = xsec.c_drainage
    DRN['h',:] = xsec.hpp

    IBOUND = xsec.gr.const(1.)


mdl = Fdm3(gr=xsec.gr, K=None, c=None, S=None, IBOUND=None, HI=None, FQ=None)
mdl.simulate(DRN=DRN, RIV=None, GHB=None, FDR=None, tm=None, htol=1e-7, maxiter=50, verbose=False):